# Core Built-in Containers (`list`, `tuple`, `dict`, `set`, `frozenset`)

Understanding the CPython internals and Big-O guarantees of Python's five core built-in containers avoids subtle performance bottlenecks and bugs.

## Big-O Cheat Sheet (Average Case in CPython)

| Operation | `list` (dynamic array) | `tuple` (fixed array) | `dict` (hash table) | `set` / `frozenset` (hash table) |
|---|---|---|---|---|
| Index / Key Lookup | $O(1)$ | $O(1)$ | $O(1)$ | N/A |
| Membership (`x in c`) | **$O(n)$** | **$O(n)$** | **$O(1)$** | **$O(1)$** |
| Append / Insert at end | $O(1)$ amortized | immutable | $O(1)$ amortized | $O(1)$ amortized |
| Insert / Delete at front | **$O(n)$** | immutable | $O(1)$ (`del d[k]`) | $O(1)$ (`discard`) |
| Slice `c[a:b]` | $O(b - a)$ | $O(b - a)$ | N/A | N/A |
| Iteration | $O(n)$ | $O(n)$ | $O(n)$ (insertion order, 3.7+) | $O(n)$ (arbitrary order) |
| Hashable? | No | Only if all elements are hashable | No | `set`: No / `frozenset`: Yes |

---
## 1. `list` vs. `tuple`

- **`list`**: over-allocated dynamic array of object pointers (resizes geometrically -> $O(1)$ amortized `.append()`).
- **`tuple`**: fixed-size array of pointers — uses less memory, slightly faster to instantiate, and **hashable** when all its items are hashable.
- **Gotcha:** Tuple immutability protects the *references*, not mutable objects inside the tuple!

In [1]:
import sys

lst = [1, 2, 3, 4, 5]
tup = (1, 2, 3, 4, 5)
print(f"Memory size -> list: {sys.getsizeof(lst)} bytes | tuple: {sys.getsizeof(tup)} bytes")

# Slicing creates a shallow copy: list[start:stop:step]
print("Reversed via slice:", lst[::-1])
print("Every 2nd element:", lst[::2])

# Hashability gotcha: tuple containing a list is NOT hashable
good_key = (1, "a", frozenset({2, 3}))
bad_key = (1, [2, 3])
print("hash(good_key):", hash(good_key))
try:
    hash(bad_key)
except TypeError as e:
    print("hash(bad_key) -> TypeError:", e)

Memory size -> list: 104 bytes | tuple: 80 bytes
Reversed via slice: [5, 4, 3, 2, 1]
Every 2nd element: [1, 3, 5]
hash(good_key): 9161037935688289719
hash(bad_key) -> TypeError: unhashable type: 'list'


---
## 2. `dict` — Compact Hash Tables (Python 3.6+ / Guaranteed Ordered in 3.7+)

Since Python 3.6, `dict` uses a **compact hash table** separating a dense entries array from a sparse indices table — cutting memory usage by ~20–25% and preserving insertion order.

Key idiomatic methods:
- `d.get(k, default)` — safe lookup without `KeyError`
- `d.setdefault(k, default)` — insert default if missing, return value (note: `default` is evaluated eagerly — prefer `defaultdict` in hot loops!)
- `d1 | d2` and `d1 |= d2` — clean dictionary merge operators (Python 3.9+)
- Dictionary views (`d.keys()`, `d.values()`, `d.items()`) are **dynamic $O(1)$ views** (not copies) and `keys()` / `items()` support set operations (`&`, `|`, `-`)!

In [2]:
base = {"lr": 1e-3, "epochs": 10, "optimizer": "adamw"}
override = {"lr": 3e-4, "warmup_steps": 500}

merged = base | override  # right-hand side wins on duplicate keys
print("Merged (|):", merged)

# Dynamic dict views & set-like key operations
shared_keys = base.keys() & override.keys()
only_in_base = base.keys() - override.keys()
print("Shared keys:", shared_keys, "| Only in base:", only_in_base)

# Safe iteration while deleting: iterate over list(d.keys()) or build a comprehension
scores = {"a": 0.9, "b": 0.3, "c": 0.85, "d": 0.2}
filtered = {k: v for k, v in scores.items() if v >= 0.5}
print("Filtered dict:", filtered)

Merged (|): {'lr': 0.0003, 'epochs': 10, 'optimizer': 'adamw', 'warmup_steps': 500}
Shared keys: {'lr'} | Only in base: {'optimizer', 'epochs'}
Filtered dict: {'a': 0.9, 'c': 0.85}


---
## 3. `set` and `frozenset`

Sets provide $O(1)$ average-case membership testing and clean mathematical set algebra:
- Union: `A | B`
- Intersection: `A & B`
- Difference: `A - B`
- Symmetric difference: `A ^ B`
- Subset / Superset: `A <= B`, `A >= B`
- `frozenset` is immutable and **hashable** (can be a `dict` key or element of another `set`).

In [3]:
ml_stack = {"python", "pytorch", "cuda", "docker", "uv"}
backend_stack = {"python", "docker", "postgres", "redis"}

print("Intersection (&):", ml_stack & backend_stack)
print("Union (|):", ml_stack | backend_stack)
print("ML only (-):", ml_stack - backend_stack)
print("Symmetric diff (^):", ml_stack ^ backend_stack)

# Order-preserving deduplication trick (Python 3.7+): dict.fromkeys()
raw = ["zurich", "berlin", "paris", "berlin", "zurich", "amsterdam"]
deduped = list(dict.fromkeys(raw))
print("Order-preserving dedup:", deduped)

# frozenset as a dict key (e.g., undirected graph edge weights)
edge_weights = {frozenset({"A", "B"}): 4.2, frozenset({"B", "C"}): 1.7}
print("Weight of edge B-A:", edge_weights[frozenset({"B", "A"})])

Intersection (&): {'docker', 'python'}
Union (|): {'cuda', 'uv', 'python', 'postgres', 'pytorch', 'redis', 'docker'}
ML only (-): {'cuda', 'pytorch', 'uv'}
Symmetric diff (^): {'cuda', 'uv', 'postgres', 'redis', 'pytorch'}
Order-preserving dedup: ['zurich', 'berlin', 'paris', 'amsterdam']
Weight of edge B-A: 4.2


---
## 4. Three Classic Python Container Gotchas

1. **Mutable default arguments** (`def f(x, acc=[]):`) — default objects are created **once** at function definition time, not per call! Use `acc=None`.
2. **Shallow vs. deep copy** — `list.copy()`, `lst[:]`, and `[[0]*3]*3` share nested references!
3. **List multiplication for 2D grids** — `[[0]*3]*3` creates 3 references to the **exact same** inner list!

In [4]:
import copy

# Gotcha 1: Mutable default argument
def append_buggy(val, target=[]):
    target.append(val)
    return target

def append_safe(val, target=None):
    if target is None:
        target = []
    target.append(val)
    return target

print("Buggy calls:", append_buggy(1), append_buggy(2))
print("Safe calls: ", append_safe(1), append_safe(2))

# Gotcha 2: [[0]*3]*3 shares inner list references!
bad_grid = [[0] * 3] * 3
bad_grid[0][0] = 99
print("Bad grid (all rows mutated!):", bad_grid)

good_grid = [[0] * 3 for _ in range(3)]
good_grid[0][0] = 99
print("Good grid:", good_grid)

# Gotcha 3: Shallow vs Deep copy
nested = [[1, 2], [3, 4]]
shallow = nested.copy()
deep = copy.deepcopy(nested)
nested[0].append(999)
print("Original:", nested, "| Shallow affected:", shallow, "| Deep unaffected:", deep)

Buggy calls: [1, 2] [1, 2]
Safe calls:  [1] [2]
Bad grid (all rows mutated!): [[99, 0, 0], [99, 0, 0], [99, 0, 0]]
Good grid: [[99, 0, 0], [0, 0, 0], [0, 0, 0]]
Original: [[1, 2, 999], [3, 4]] | Shallow affected: [[1, 2, 999], [3, 4]] | Deep unaffected: [[1, 2], [3, 4]]
